# Matmul ladder on a T4: siboehm's kernels 1-3 vs cuBLAS

Companion to the card [Matmul I: the naive kernel and the lower bound](https://seyonv.github.io/explainers/perf-3-kernels/matmul-naive.html).

**Runtime → Change runtime type → T4 GPU**, then Run all.

What you will measure, at 4096² and 1024² fp32 (C = A·B, alpha = 1, beta = 0):

1. Kernel 1, naive: one thread per C entry, 32×32 blocks.
2. Kernel 2, global-memory coalescing: the same math, with a warp's threads walking along a row of C.
3. Kernel 3, shared-memory cache-blocking: 32×32 tiles of A and B staged in shared memory.
4. cuBLAS (`cublasSgemm`, called from the same program) and `torch.matmul` with TF32 disabled.

Each kernel is timed with `cudaEvent` after warm-up runs and checked against cuBLAS. The program prints GFLOP/s (counted as 2·M·N·K), % of cuBLAS and % of the T4's advertised 8.1 TFLOPS fp32.

**Credit and license.** Kernels 1-3 are copied, with minor edits (bounds and comments), from Simon Boehm's [SGEMM_CUDA](https://github.com/siboehm/SGEMM_CUDA) (`src/kernels/1_naive.cuh`, `2_kernel_global_mem_coalesce.cuh`, `3_kernel_shared_mem_blocking.cuh`), MIT License, Copyright (c) 2023 Simon Boehm. The article: [How to Optimize a CUDA Matmul Kernel for cuBLAS-like Performance: a Worklog](https://siboehm.com/articles/22/CUDA-MMM).

**Status:** not yet run by the author of the card (no NVIDIA GPU). Your run is the exercise. Colab's T4 clocks vary between sessions, so expect some run-to-run spread.

In [ ]:
!nvidia-smi

In [ ]:
%%writefile ladder.cu
// Kernels 1-3 adapted from github.com/siboehm/SGEMM_CUDA (MIT License, (c) 2023 Simon Boehm).
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))
#define CUDA_CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
  printf("CUDA error %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)
#define CUBLAS_CHECK(x) do { cublasStatus_t s = (x); if (s != CUBLAS_STATUS_SUCCESS) { \
  printf("cuBLAS error %d at %s:%d\n", (int)s, __FILE__, __LINE__); exit(1); } } while (0)

// Kernel 1: naive. x walks down the rows of C, so the 32 threads of a warp
// (consecutive threadIdx.x) read 32 different rows of A: uncoalesced.
__global__ void sgemm_naive(int M, int N, int K, float alpha, const float *A,
                            const float *B, float beta, float *C) {
  const unsigned x = blockIdx.x * blockDim.x + threadIdx.x;
  const unsigned y = blockIdx.y * blockDim.y + threadIdx.y;
  if (x < M && y < N) {
    float tmp = 0.0f;
    for (int i = 0; i < K; ++i) tmp += A[x * K + i] * B[i * N + y];
    C[x * N + y] = alpha * tmp + beta * C[x * N + y];
  }
}

// Kernel 2: same math, but consecutive threads get consecutive columns of C,
// so a warp's reads of B (and writes of C) are consecutive: coalesced.
template <const unsigned BLOCKSIZE>
__global__ void sgemm_global_mem_coalesce(int M, int N, int K, float alpha,
                                          const float *A, const float *B,
                                          float beta, float *C) {
  const int cRow = blockIdx.x * BLOCKSIZE + (threadIdx.x / BLOCKSIZE);
  const int cCol = blockIdx.y * BLOCKSIZE + (threadIdx.x % BLOCKSIZE);
  if (cRow < M && cCol < N) {
    float tmp = 0.0f;
    for (int i = 0; i < K; ++i) tmp += A[cRow * K + i] * B[i * N + cCol];
    C[cRow * N + cCol] = alpha * tmp + beta * C[cRow * N + cCol];
  }
}

// Kernel 3: shared-memory cache-blocking. Assumes M, N, K are multiples of BLOCKSIZE.
template <const int BLOCKSIZE>
__global__ void sgemm_shared_mem_block(int M, int N, int K, float alpha,
                                       const float *A, const float *B,
                                       float beta, float *C) {
  const unsigned cRow = blockIdx.x;
  const unsigned cCol = blockIdx.y;
  __shared__ float As[BLOCKSIZE * BLOCKSIZE];
  __shared__ float Bs[BLOCKSIZE * BLOCKSIZE];
  const unsigned threadCol = threadIdx.x % BLOCKSIZE;
  const unsigned threadRow = threadIdx.x / BLOCKSIZE;
  A += cRow * BLOCKSIZE * K;
  B += cCol * BLOCKSIZE;
  C += cRow * BLOCKSIZE * N + cCol * BLOCKSIZE;
  float tmp = 0.0f;
  for (int bkIdx = 0; bkIdx < K; bkIdx += BLOCKSIZE) {
    As[threadRow * BLOCKSIZE + threadCol] = A[threadRow * K + threadCol];
    Bs[threadRow * BLOCKSIZE + threadCol] = B[threadRow * N + threadCol];
    __syncthreads();
    A += BLOCKSIZE;
    B += BLOCKSIZE * N;
    for (int dotIdx = 0; dotIdx < BLOCKSIZE; ++dotIdx)
      tmp += As[threadRow * BLOCKSIZE + dotIdx] * Bs[dotIdx * BLOCKSIZE + threadCol];
    __syncthreads();
  }
  C[threadRow * N + threadCol] = alpha * tmp + beta * C[threadRow * N + threadCol];
}

// Row-major C = A*B through column-major cuBLAS: compute C^T = B^T * A^T.
void run_cublas(cublasHandle_t h, int M, int N, int K, float alpha, const float *A,
                const float *B, float beta, float *C) {
  CUBLAS_CHECK(cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, B, N, A, K, &beta, C, N));
}

void run_kernel(int k, cublasHandle_t h, int M, int N, int K, float alpha,
                const float *A, const float *B, float beta, float *C) {
  if (k == 0) { run_cublas(h, M, N, K, alpha, A, B, beta, C); return; }
  if (k == 1) {
    dim3 grid(CEIL_DIV(M, 32), CEIL_DIV(N, 32)), block(32, 32);
    sgemm_naive<<<grid, block>>>(M, N, K, alpha, A, B, beta, C);
  } else if (k == 2) {
    dim3 grid(CEIL_DIV(M, 32), CEIL_DIV(N, 32)), block(32 * 32);
    sgemm_global_mem_coalesce<32><<<grid, block>>>(M, N, K, alpha, A, B, beta, C);
  } else {
    dim3 grid(CEIL_DIV(M, 32), CEIL_DIV(N, 32)), block(32 * 32);
    sgemm_shared_mem_block<32><<<grid, block>>>(M, N, K, alpha, A, B, beta, C);
  }
  CUDA_CHECK(cudaGetLastError());
}

int main() {
  const char *names[] = {"cuBLAS", "1 naive", "2 GMEM coalescing", "3 SMEM caching"};
  const int sizes[] = {1024, 4096};
  const double T4_PEAK = 8.1e12;
  cublasHandle_t h;
  CUBLAS_CHECK(cublasCreate(&h));
  FILE *csv = fopen("ladder.csv", "w");
  fprintf(csv, "n,kernel,ms,gflops\n");
  for (int n : sizes) {
    const int M = n, N = n, K = n;
    if (n % 32 != 0) { printf("kernel 3 needs n %% 32 == 0\n"); return 1; }
    size_t bytes = (size_t)n * n * sizeof(float);
    std::vector<float> hA((size_t)n * n), hB((size_t)n * n), hRef((size_t)n * n), hC((size_t)n * n);
    srand(0);
    for (auto &v : hA) v = (float)rand() / RAND_MAX * 2.f - 1.f;
    for (auto &v : hB) v = (float)rand() / RAND_MAX * 2.f - 1.f;
    float *A, *B, *C, *Cref;
    CUDA_CHECK(cudaMalloc(&A, bytes)); CUDA_CHECK(cudaMalloc(&B, bytes));
    CUDA_CHECK(cudaMalloc(&C, bytes)); CUDA_CHECK(cudaMalloc(&Cref, bytes));
    CUDA_CHECK(cudaMemcpy(A, hA.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(B, hB.data(), bytes, cudaMemcpyHostToDevice));
    const float alpha = 1.f, beta = 0.f;  // beta = 0: C must still hold finite values, so zero it

    CUDA_CHECK(cudaMemset(Cref, 0, bytes));
    run_cublas(h, M, N, K, alpha, A, B, beta, Cref);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(hRef.data(), Cref, bytes, cudaMemcpyDeviceToHost));

    const double flops = 2.0 * M * N * K;
    const int reps = n >= 4096 ? 5 : 20;
    double cublas_gf = 0;
    printf("\n== n = %d  (2*n^3 = %.2f GFLOP, %d timed runs after 2 warm-ups) ==\n", n, flops / 1e9, reps);
    printf("%-20s %10s %10s %10s %10s %10s\n", "kernel", "ms", "GFLOP/s", "% cuBLAS", "% 8.1 TF", "max err");
    for (int k : {0, 1, 2, 3}) {
      // correctness run
      CUDA_CHECK(cudaMemset(C, 0, bytes));
      run_kernel(k, h, M, N, K, alpha, A, B, beta, C);
      CUDA_CHECK(cudaDeviceSynchronize());
      CUDA_CHECK(cudaMemcpy(hC.data(), C, bytes, cudaMemcpyDeviceToHost));
      double maxerr = 0;
      for (size_t i = 0; i < hC.size(); ++i) maxerr = fmax(maxerr, fabs((double)hC[i] - hRef[i]));
      // warm-up, then timed runs
      for (int w = 0; w < 2; ++w) run_kernel(k, h, M, N, K, alpha, A, B, beta, C);
      cudaEvent_t t0, t1;
      CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
      CUDA_CHECK(cudaEventRecord(t0));
      for (int r = 0; r < reps; ++r) run_kernel(k, h, M, N, K, alpha, A, B, beta, C);
      CUDA_CHECK(cudaEventRecord(t1));
      CUDA_CHECK(cudaEventSynchronize(t1));
      float ms_total;
      CUDA_CHECK(cudaEventElapsedTime(&ms_total, t0, t1));
      double ms = ms_total / reps, gf = flops / (ms * 1e-3) / 1e9;
      if (k == 0) cublas_gf = gf;
      printf("%-20s %10.3f %10.1f %9.1f%% %9.1f%% %10.2e %s\n", names[k], ms, gf, gf / cublas_gf * 100,
             gf * 1e9 / T4_PEAK * 100, maxerr, maxerr < 1e-2 ? "ok" : "MISMATCH");
      fprintf(csv, "%d,%s,%.4f,%.2f\n", n, names[k], ms, gf);
      CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    }
    CUDA_CHECK(cudaFree(A)); CUDA_CHECK(cudaFree(B)); CUDA_CHECK(cudaFree(C)); CUDA_CHECK(cudaFree(Cref));
  }
  fclose(csv);
  CUBLAS_CHECK(cublasDestroy(h));
  return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o ladder ladder.cu -lcublas && ./ladder

## cuBLAS through PyTorch, TF32 off

`torch.matmul` on fp32 CUDA tensors calls cuBLAS. TF32 is switched off so it is a true fp32 SGEMM (the T4 is Turing, sm_75, which has no TF32 tensor cores anyway). This cell times it the same way and puts every kernel from the CSV next to it.

In [ ]:
import csv, torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU"
print(torch.cuda.get_device_name(0))

T4_PEAK = 8.1e12
rows = list(csv.DictReader(open("ladder.csv")))
for n in (1024, 4096):
    a = torch.rand(n, n, device="cuda") * 2 - 1
    b = torch.rand(n, n, device="cuda") * 2 - 1
    for _ in range(3):                      # warm-up
        c = torch.matmul(a, b)
    reps = 5 if n >= 4096 else 20
    t0, t1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    t0.record()
    for _ in range(reps):
        c = torch.matmul(a, b)
    t1.record(); torch.cuda.synchronize()
    ms = t0.elapsed_time(t1) / reps
    torch_gf = 2 * n**3 / (ms * 1e-3) / 1e9
    ref64 = (a.double() @ b.double()).float()
    print(f"\nn = {n}: torch.matmul {ms:.3f} ms, {torch_gf:.1f} GFLOP/s = {torch_gf*1e9/T4_PEAK*100:.1f}% of 8.1 TF, "
          f"max err vs fp64 {(c - ref64).abs().max().item():.2e}")
    print(f"{'kernel':20s} {'GFLOP/s':>10} {'% torch.matmul':>15} {'% 8.1 TF':>10}")
    for r in rows:
        if int(r["n"]) == n:
            g = float(r["gflops"])
            print(f"{r['kernel']:20s} {g:10.1f} {g/torch_gf*100:14.1f}% {g*1e9/T4_PEAK*100:9.1f}%")

## Reference numbers (siboehm, RTX A6000, 4092², fp32)

| Kernel | GFLOP/s | % of cuBLAS |
|---|---|---|
| 1: Naive | 309.0 | 1.3% |
| 2: GMEM coalescing | 1986.5 | 8.5% |
| 3: SMEM caching | 2980.3 | 12.8% |
| 0: cuBLAS | 23249.6 | 100% |

The A6000 is advertised at 30 TFLOPS fp32 and 768 GB/s; the T4 at 8.1 TFLOPS and 320 GB/s, so expect every absolute number to be lower. Compare the **ratios** (kernel 2 over kernel 1, kernel 3 over kernel 2, each over cuBLAS), not the GFLOP/s.

Lower bound for your run (our recomputation, `labs/matmul-bounds.py`): at 4096², 2·n³ = 137.4 GFLOP takes at least 16.97 ms at 8.1 TFLOPS, while the minimum 268 MB of traffic takes 0.84 ms at 320 GB/s.

## Try this
1. Change `sizes` to `{512, 1024, 2048, 4096}` and watch where the gap between kernel 3 and cuBLAS is smallest.
2. Profile one kernel: `!ncu --kernel-name sgemm_naive --launch-count 1 --section MemoryWorkloadAnalysis ./ladder` and compare its DRAM throughput with siboehm's "15 GB/s" on the A6000 (Colab may not permit `ncu` counters; if it refuses, that is the reason).
3. In kernel 1, swap the roles of `x` and `y` (so consecutive threads walk along a row of C) and confirm it matches kernel 2's speed.